# 11. ivis variants

Produces, from the saved results of the ivis variants (ivis retrained so that its similarity learning carries the
decision, and a nearest-neighbour vote on its embedding):

- **Table S-XXV** (`runs_ivisvar/report/clean.csv`, `worst_case.csv`): clean AUPRC and worst-case ASR at eps = 0.25
  under the constrained (C), consistent (K) and unconstrained (U) attackers.
- **Table S-XXVI** (`runs_ivisvar/report/paired_holm.csv`): paired differences against the MLP and the study's ivis,
  Holm-adjusted.
- **Table S-XXVII** (`runs_ivisvar/report/geometry.csv`): embedding geometry of each variant.

Inputs: `runs_ivisvar/<d>/` (the variants' models and attack outputs) and its summaries `runs_ivisvar/report/`
(`worst_case.csv`, `paired.csv`, `clean.csv` from `ivis_variants.py report`; `geometry.csv` from `ivis_variants.py diag`),
plus the study's grid `runs/<d>/attacks/per_example_full_*.npz`, `runs_consistent/<d>/attacks/per_example_consistent*.npz`
and the reference models' `metrics.json`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))   # the library (not needed after pip install -e .)
from fraudadv import notebook as nb
ROOT = nb.setup()          # works from the folder root, sets the study's environment

## Regenerate the summaries in `runs_ivisvar/report/`

`scripts/analysis/ivis_variants.py report` rebuilds `worst_case.csv` (worst-case ASR per model, setting and budget),
`paired.csv` (exact McNemar against the study's MLP, ivis and FT-Transformer) and `clean.csv` (clean metrics) from the
per-example outcomes. It overwrites the saved copies, so the cell keeps their bytes first and checks the rebuilt files
are identical. (`geometry.csv` comes from `diag`, which loads every encoder; it is not rebuilt here.) About half a minute.

In [ ]:
import os
import runpy
import subprocess

REPORT = ROOT / "runs_ivisvar" / "report"
SUMMARIES = ["worst_case.csv", "paired.csv", "clean.csv"]
saved = {f: (REPORT / f).read_bytes() for f in SUMMARIES}

done = subprocess.run([sys.executable, str(ROOT / "scripts/analysis/ivis_variants.py"), "report"], cwd=ROOT,
                      env=dict(os.environ, PYTHONPATH=str(ROOT / "src")), capture_output=True, text=True, check=True)
print("\n".join(done.stdout.splitlines()[:20]), "\n...")
for f in SUMMARIES:
    print(f"{f:>16}: rebuilt file", "identical to the saved one" if (REPORT / f).read_bytes() == saved[f] else "DIFFERENT")

## Tables S-XXV to S-XXVII

Table S-XXV and Table S-XXVII are the summaries above (`clean.csv`, `worst_case.csv`, `geometry.csv`).
`scripts/analysis/ivis_variants_paired.py` keeps the comparisons of `paired.csv` against the MLP and the study's ivis
and adds the difference in ASR and Holm-adjusted p-values (`runs_ivisvar/report/paired_holm.csv`, Table S-XXVI), as
`scripts/make_results.sh` runs it.

In [ ]:
W = nb.show_csv("runs_ivisvar/report/worst_case.csv", rows=5)
display(W[W.eps == 0.25].pivot_table(index=["dataset", "mode"], columns="model", values="asr").round(3))

In [ ]:
nb.show_csv("runs_ivisvar/report/clean.csv", rows=8);

In [ ]:
runpy.run_path(str(ROOT / "scripts/analysis/ivis_variants_paired.py"), run_name="__main__")
nb.show_csv("runs_ivisvar/report/paired_holm.csv", rows=12);

In [ ]:
nb.show_csv("runs_ivisvar/report/geometry.csv", rows=8);

## Rerun the runs behind these results from scratch

The commands of `scripts/analysis/ivis_variants.py` (the ivis variants are not part of `scripts/reproduce.sh`), for one
dataset of the four it ran on (Sparkov, TabFormer, BAF, IEEE-CIS; not ULB or PaySim): tune the two retrained variants
(10 TPE trials each), train all three with three seeds, attack the five models with the study's grid and the neighbour
votes with the adaptive attacks, and, where the consistent attacker differs from the constrained one (Sparkov,
IEEE-CIS), the same attacks with `--modes consistent`. `diag` and `report` then summarise all four datasets. They take
hours and write into `runs_ivisvar/`; they skip work whose outputs exist. Not run here.

In [ ]:
RERUN = False
DATASET = "sparkov"
if RERUN:
    def ivis_variants(*args):
        """python3 scripts/analysis/ivis_variants.py <args>, from the folder root."""
        subprocess.run([sys.executable, str(ROOT / "scripts/analysis/ivis_variants.py"), *args], check=True, cwd=ROOT,
                       env=dict(os.environ, PYTHONPATH=str(ROOT / "src")))

    for variant in ("sup", "smooth"):
        ivis_variants("tune", DATASET, variant)
    for variant in ("base", "sup", "smooth"):
        ivis_variants("train", DATASET, variant)
    ivis_variants("attack", DATASET)                   # the five models, unconstrained and constrained
    ivis_variants("adaptive", DATASET)                 # the three neighbour votes
    if DATASET in ("sparkov", "ieee_cis"):             # consistent attacker (elsewhere it equals the constrained one)
        ivis_variants("attack", DATASET, "--modes", "consistent")
        ivis_variants("adaptive", DATASET, "--modes", "consistent")
    ivis_variants("diag")                              # geometry.csv, all four datasets
    ivis_variants("report")                            # worst_case.csv, paired.csv, clean.csv

## Check against the reference build

The Holm-adjusted comparisons this notebook wrote (the rebuilt `runs_ivisvar/report/` summaries were checked above),
compared byte for byte with the same path under `$FRAUDADV_REFERENCE`, when the reference holds it.

In [ ]:
nb.check(["runs_ivisvar/report/paired_holm.csv"])